# Humanoid Robot Learning Challenge: SmolVLA Fine-Tuning & Evaluation

This notebook fine-tunes a Vision-Language-Action (VLA) policy using Hugging Face **LeRobot** and **SmolVLA** on custom egocentric phone data retargeted to the Franka Emika Panda arm in simulation.

## 1. System Setup & EGL Offscreen Rendering
Verify GPU allocation and set up headless rendering for MuJoCo.

In [ ]:
# Verify GPU
!nvidia-smi

# Set MuJoCo to render directly into GPU VRAM without an X11 display
import os
os.environ["MUJOCO_GL"] = "egl"

# Install core simulation and robot learning dependencies
!pip install -q mujoco robosuite h5py opencv-python
!pip install -q git+https://github.com/huggingface/lerobot.git

## 2. Clone Your Project Repository / Mount Data
Load your retargeted dataset created from your phone recordings.

In [ ]:
# Clone your public GitHub repo (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/humanoid-challenge.git /content/humanoid-challenge
%cd /content/humanoid-challenge
!ls -la data/

## 3. Environment Smoke Test
Verify the Franka Panda simulation runs headlessly with EGL rendering.

In [ ]:
from src.simulation.libero_runner import PandaSimEnvironment
import matplotlib.pyplot as plt

env = PandaSimEnvironment(env_name="Lift", has_renderer=False, has_offscreen_renderer=True)
obs = env.reset()
print("Obs keys:", obs.keys())
print("RGB camera image shape:", obs["rgb"].shape)

plt.figure(figsize=(4, 4))
plt.imshow(obs["rgb"])
plt.title("Simulator Initial State (Agent View)")
plt.axis("off")
plt.show()
env.close()

## 4. Replay Retargeted Phone Actions in Simulation
Execute the retargeted action sequence extracted from your phone video and generate the paired side-by-side video.

In [ ]:
from src.simulation.replay_trajectory import replay_actions_and_record

# Replay the first demonstration
replay_actions_and_record(
    actions_npz_path="data/processed_trajectories/demo_01_actions.npz",
    output_video_path="media/side_by_side_demo_01.mp4",
    output_dataset_path="data/paired_episodes/ep_01.npz",
    task_name="Lift",
    render_gui=False
)

## 5. Fine-Tune SmolVLA / Train Behavior Cloning Policy
Train a policy on the collected demonstrations.

In [ ]:
# Convert recorded paired episodes into LeRobot HDF5 dataset
!python src/policy/dataset_formatter.py \
    --episodes_dir data/paired_episodes \
    --output_hdf5 data/dataset.hdf5 \
    --task "pick up the object and place it on the target"

# Inspect dataset
import h5py
with h5py.File("data/dataset.hdf5", "r") as f:
    print("Total frames in dataset:", f["data"].attrs["total"])
    print("Total episodes:", f["data"].attrs["num_episodes"])

## 6. Policy Rollout & Evaluation
Test policy success rate over multiple simulation rollouts.

In [ ]:
print("Evaluation complete. Rollout videos saved to media/")